# Re-Ranking in RAG: A Real-World Use Case

## Problem: Customer Support Knowledge Base Search

Imagine you're building a customer support chatbot for a tech company. Users ask questions, and the system retrieves relevant articles from a knowledge base. The challenge: **basic semantic search often retrieves documents that are *somewhat* related but not the best answer**.

Re-ranking solves this by taking the initial retrieved documents and re-scoring them with a more powerful model to push the most relevant ones to the top.

### How It Works (Step by Step)

1. **Embed & Store** - We embed knowledge base articles into vectors
2. **Retrieve** - Given a user query, retrieve top-K candidates using cosine similarity
3. **Re-Rank** - Use a cross-encoder (via OpenAI) to re-score each candidate against the query
4. **Return** - Present the re-ranked results (better ordering = better answers)

---

## Step 1: Install Dependencies

In [ ]:
!pip install openai numpy -q

## Step 2: Setup OpenAI Client & API Key

In [ ]:
import os
import numpy as np
from openai import OpenAI

# Set your OpenAI API key here
os.environ["OPENAI_API_KEY"] = "sk-your-api-key-here"  # <-- Replace with your key

client = OpenAI()

## Step 3: Create a Simulated Knowledge Base

These represent real support articles a customer might search through. Some are closely related to a query but not the best answer — this is where re-ranking shines.

In [ ]:
# Simulated knowledge base: tech support articles
knowledge_base = [
    {
        "id": 1,
        "title": "How to Reset Your Password",
        "content": "To reset your password, go to Settings > Account > Security > Reset Password. "
                   "You'll receive a verification email. Click the link and enter your new password. "
                   "Make sure it's at least 8 characters with a mix of letters and numbers."
    },
    {
        "id": 2,
        "title": "Account Locked After Multiple Failed Attempts",
        "content": "If your account is locked after too many failed login attempts, wait 30 minutes "
                   "and try again. If you still can't log in, contact support with your registered "
                   "email address. We'll verify your identity and unlock your account manually."
    },
    {
        "id": 3,
        "title": "Two-Factor Authentication Setup",
        "content": "Enable 2FA by going to Settings > Security > Two-Factor Authentication. "
                   "You can use an authenticator app like Google Authenticator or receive SMS codes. "
                   "2FA adds an extra layer of security to prevent unauthorized access."
    },
    {
        "id": 4,
        "title": "Billing and Subscription Management",
        "content": "Manage your subscription from Settings > Billing. You can upgrade, downgrade, "
                   "or cancel your plan. Refunds are available within 14 days of purchase. "
                   "For billing disputes, email billing@company.com with your invoice number."
    },
    {
        "id": 5,
        "title": "App Crashing on Startup",
        "content": "If the app crashes on startup, try clearing the cache: Settings > App > Clear Cache. "
                   "If that doesn't work, uninstall and reinstall the latest version. "
                   "Make sure your device OS is updated to the minimum supported version."
    },
    {
        "id": 6,
        "title": "How to Change Your Email Address",
        "content": "To change your email, go to Settings > Account > Email. Enter your new email "
                   "and confirm with your current password. A verification link will be sent to "
                   "the new email. You must click it within 24 hours or the change will expire."
    },
    {
        "id": 7,
        "title": "Password Requirements and Best Practices",
        "content": "Your password must be at least 8 characters and include uppercase letters, "
                   "lowercase letters, and numbers. We recommend using a password manager. "
                   "Avoid reusing passwords across different services for better security."
    },
    {
        "id": 8,
        "title": "Forgot Username Recovery",
        "content": "If you forgot your username, click 'Forgot Username' on the login page. "
                   "Enter the email address associated with your account. Your username will "
                   "be sent to that email within a few minutes."
    }
]

print(f"Knowledge base loaded: {len(knowledge_base)} articles")

## Step 4: Generate Embeddings for the Knowledge Base

We use OpenAI's embedding model to convert each article into a vector. This lets us do semantic similarity search.

In [ ]:
def get_embeddings(texts, model="text-embedding-3-small"):
    """Get embeddings for a list of texts using OpenAI."""
    response = client.embeddings.create(input=texts, model=model)
    return [item.embedding for item in response.data]


# Embed all articles (combining title + content for richer representation)
article_texts = [f"{doc['title']}. {doc['content']}" for doc in knowledge_base]
article_embeddings = get_embeddings(article_texts)

print(f"Generated embeddings for {len(article_embeddings)} articles")
print(f"Embedding dimension: {len(article_embeddings[0])}")

## Step 5: Retrieval — Find Top-K Candidates via Cosine Similarity

This is the standard RAG retrieval step. We compute cosine similarity between the user query and all stored articles, then return the top-K most similar ones.

In [ ]:
def cosine_similarity(a, b):
    """Compute cosine similarity between two vectors."""
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


def retrieve(query, top_k=5):
    """Retrieve top-K articles by cosine similarity."""
    query_embedding = get_embeddings([query])[0]
    
    scores = []
    for i, doc_emb in enumerate(article_embeddings):
        sim = cosine_similarity(query_embedding, doc_emb)
        scores.append((i, sim))
    
    # Sort by similarity (highest first)
    scores.sort(key=lambda x: x[1], reverse=True)
    
    results = []
    for idx, sim in scores[:top_k]:
        results.append({
            "document": knowledge_base[idx],
            "similarity_score": round(sim, 4)
        })
    
    return results


# Test retrieval
query = "I can't log into my account, it says too many attempts"
retrieved = retrieve(query, top_k=5)

print(f"Query: '{query}'\n")
print("=" * 60)
print("INITIAL RETRIEVAL (before re-ranking):")
print("=" * 60)
for i, result in enumerate(retrieved, 1):
    print(f"\n#{i} [Score: {result['similarity_score']}]")
    print(f"   Title: {result['document']['title']}")
    print(f"   Content: {result['document']['content'][:100]}...")

## Step 6: Re-Ranking with OpenAI

Here's the key innovation. We use OpenAI's GPT model as a **cross-encoder** — it looks at each (query, document) pair together and scores how well the document answers the query.

This is more powerful than embedding similarity because:
- Embeddings compare texts independently (bi-encoder)
- Re-ranking compares them jointly, understanding nuanced relevance

In [ ]:
import json


def rerank(query, documents):
    """
    Re-rank documents using OpenAI as a cross-encoder.
    Asks the model to score each document's relevance to the query.
    """
    # Build the prompt with all candidate documents
    doc_list = ""
    for i, doc in enumerate(documents):
        doc_list += f"\nDocument {i+1}:\nTitle: {doc['document']['title']}\nContent: {doc['document']['content']}\n"
    
    prompt = f"""You are a relevance scoring system. Given a user query and a list of documents, 
score each document's relevance to the query on a scale of 1-10.

Consider:
- Does the document directly answer the user's question?
- Is the information actionable for the user's specific situation?
- How well does the document match the user's intent?

User Query: "{query}"

Documents:
{doc_list}

Return a JSON array with objects containing "document_index" (1-based) and "relevance_score" (1-10).
Also include a brief "reason" for each score.

Return ONLY valid JSON, no other text."""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    
    # Parse the response
    result_text = response.choices[0].message.content.strip()
    # Remove markdown code fences if present
    if result_text.startswith("```"):
        result_text = result_text.split("\n", 1)[1].rsplit("```", 1)[0]
    
    scores = json.loads(result_text)
    
    # Attach re-rank scores to documents
    reranked = []
    for score_item in scores:
        idx = score_item["document_index"] - 1
        reranked.append({
            "document": documents[idx]["document"],
            "original_similarity": documents[idx]["similarity_score"],
            "rerank_score": score_item["relevance_score"],
            "reason": score_item.get("reason", "")
        })
    
    # Sort by re-rank score (highest first)
    reranked.sort(key=lambda x: x["rerank_score"], reverse=True)
    
    return reranked


print("Re-ranking function defined. Ready for Step 7.")

## Step 7: Compare Results — Before vs After Re-Ranking

Let's run the full pipeline and see how re-ranking improves the order of results.

In [ ]:
# Run re-ranking on our retrieved results
query = "I can't log into my account, it says too many attempts"
retrieved = retrieve(query, top_k=5)
reranked = rerank(query, retrieved)

print(f"Query: '{query}'")
print("\n" + "=" * 60)
print("BEFORE RE-RANKING (cosine similarity only):")
print("=" * 60)
for i, result in enumerate(retrieved, 1):
    print(f"  #{i} [Similarity: {result['similarity_score']}] {result['document']['title']}")

print("\n" + "=" * 60)
print("AFTER RE-RANKING (with GPT cross-encoder):")
print("=" * 60)
for i, result in enumerate(reranked, 1):
    print(f"  #{i} [Re-rank: {result['rerank_score']}/10 | Original sim: {result['original_similarity']}]")
    print(f"      Title: {result['document']['title']}")
    print(f"      Reason: {result['reason']}")
    print()

## Step 8: Try More Queries to See the Difference

Let's test with different queries to highlight when re-ranking makes the biggest impact.

In [ ]:
test_queries = [
    "How do I make my account more secure?",
    "The app won't open on my phone",
    "I want a refund for my subscription",
]

for query in test_queries:
    print("\n" + "#" * 70)
    print(f"QUERY: '{query}'")
    print("#" * 70)
    
    retrieved = retrieve(query, top_k=4)
    reranked = rerank(query, retrieved)
    
    print("\n  BEFORE (similarity):")
    for i, r in enumerate(retrieved, 1):
        print(f"    #{i} [{r['similarity_score']}] {r['document']['title']}")
    
    print("\n  AFTER (re-ranked):")
    for i, r in enumerate(reranked, 1):
        print(f"    #{i} [Score: {r['rerank_score']}/10] {r['document']['title']}")
        print(f"        → {r['reason']}")

## Step 9: Full RAG Pipeline with Re-Ranking — Generate Final Answer

Now let's put it all together: retrieve → re-rank → generate a response using the best document.

In [ ]:
def rag_with_reranking(query):
    """
    Full RAG pipeline:
    1. Retrieve top candidates
    2. Re-rank them
    3. Generate answer using the top re-ranked document
    """
    # Step 1: Retrieve
    retrieved = retrieve(query, top_k=5)
    
    # Step 2: Re-rank
    reranked = rerank(query, retrieved)
    
    # Step 3: Use top result as context for answer generation
    best_doc = reranked[0]["document"]
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": "You are a helpful customer support agent. Answer the user's question "
                           "based on the provided knowledge base article. Be concise and friendly."
            },
            {
                "role": "user",
                "content": f"User question: {query}\n\n"
                           f"Relevant article:\nTitle: {best_doc['title']}\n"
                           f"Content: {best_doc['content']}"
            }
        ],
        temperature=0.3
    )
    
    answer = response.choices[0].message.content
    
    return {
        "query": query,
        "source_article": best_doc["title"],
        "rerank_score": reranked[0]["rerank_score"],
        "answer": answer
    }


# Test the full pipeline
result = rag_with_reranking("I can't log into my account, it says too many attempts")

print(f"Question: {result['query']}")
print(f"\nSource: {result['source_article']} (relevance: {result['rerank_score']}/10)")
print(f"\nAnswer:\n{result['answer']}")

In [ ]:
# Try another question
result = rag_with_reranking("How do I make my account more secure?")

print(f"Question: {result['query']}")
print(f"\nSource: {result['source_article']} (relevance: {result['rerank_score']}/10)")
print(f"\nAnswer:\n{result['answer']}")

## Summary: Why Re-Ranking Matters

| Aspect | Without Re-Ranking | With Re-Ranking |
|--------|-------------------|------------------|
| Retrieval | Cosine similarity (fast, approximate) | Same initial retrieval |
| Scoring | Independent embedding comparison | Joint query-document understanding |
| Quality | Good recall, mediocre precision | Better precision at top positions |
| Latency | Fast | Adds ~1 second for GPT call |
| Cost | Embedding API only | + LLM API call for re-ranking |

### When to Use Re-Ranking:
- When the top-1 result matters most (chatbots, search)
- When your initial retrieval returns "close but not quite" results
- When queries are ambiguous or multi-intent
- When you have compute budget for an extra LLM call

### Real-World Applications:
- Customer support bots (this demo)
- Legal document search
- Medical knowledge retrieval
- E-commerce product search
- Internal company wiki search